# Stone Pipeline on Colab GPU
Run all cells in order.

In [ ]:
# 路径与参数配置：支持填写“具体文件”或“文件夹路径”（会自动识别）
# 下面默认值对齐 ONC-MAPPING/src/pipeline_config.py 与 gui_app.py
DRIVE_ROOT = '/content/drive/MyDrive/ONCMAPPING/DEPLOY'
ORTHO = f'{DRIVE_ROOT}/ORIGINAL_IMG'
CANOPY = f'{DRIVE_ROOT}/CANOPY_IMAGE'
VEG = f'{DRIVE_ROOT}/VEGE_MAP'
WEIGHTS = f'{DRIVE_ROOT}/best.pt'
OUT_DIR = f'{DRIVE_ROOT}/OUT'
# 检测参数（建议默认）
TILE_SIZE = 512            # 建议: 512；显存紧张可 384，细节优先可 640/768
OVERLAP = 128              # 建议: 128；通常设置为 TILE_SIZE 的 1/4 左右
CONF = 0.25                # 建议: 0.25；漏检多可降到 0.20，误检多可升到 0.30+
IOU_NMS = 0.35             # 建议: 0.35；对齐 src 默认
GRID_SIZE_M = 10.0
MAX_TILES = None           # 建议: None 全量；调试可设 5/20
# 栖息地评分参数（建议默认，对齐 src）
CANOPY_OVERLAP_THRESHOLD = 0.2
VEGETATION_WEIGHT = 0.7
ROCK_WEIGHT = 0.3
ROCK_PERCENTILE = 95.0
SMOKE_TEST = False

In [ ]:
# 依赖安装：安装地理数据处理和 YOLO 推理所需库
!pip -q install geopandas rasterio shapely fiona pyproj rtree ultralytics pandas numpy

In [ ]:
# 挂载云盘：将 Google Drive 挂载到 /content/drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# GPU 环境检查：确认 PyTorch 版本、CUDA 是否可用和显卡名称
import torch
print(torch.__version__)
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu_name =', torch.cuda.get_device_name(0))

In [ ]:
# 主算法代码：定义从切片检测到评分与输出的完整流程
import os
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from rasterio.transform import xy
from shapely.geometry import box
from ultralytics import YOLO

# GDAL 兜底配置：当 .shx 缺失时尝试自动恢复（仍建议上传完整 shp 组件）
os.environ['SHAPE_RESTORE_SHX'] = 'YES'

# 日志工具：输出带时间戳的运行信息
def log(message):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)

# 输出目录准备：确保输出目录存在
def ensure_workspace(out_dir):
    os.makedirs(out_dir, exist_ok=True)
    log(f"Output directory is ready: {out_dir}")

# 像元归一化：将任意数值范围拉伸到 0-255
def normalize_to_uint8(arr):
    arr = arr.astype(np.float32)
    min_v = float(np.nanmin(arr))
    max_v = float(np.nanmax(arr))
    if max_v <= min_v:
        return np.zeros(arr.shape, dtype=np.uint8)
    out = (arr - min_v) / (max_v - min_v) * 255.0
    return out.astype(np.uint8)

# 影像切片读取：按窗口读取并标准化成 3 通道 RGB
def read_rgb_tile(src, x_off, y_off, width, height):
    window = Window(x_off, y_off, width, height)
    band_count = min(src.count, 3)
    arr = src.read(indexes=list(range(1, band_count + 1)), window=window)
    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=0)
    if arr.shape[0] == 1:
        arr = np.repeat(arr, 3, axis=0)
    if arr.shape[0] == 2:
        arr = np.concatenate([arr, arr[:1]], axis=0)
    arr = arr[:3]
    rgb = np.transpose(arr, (1, 2, 0))
    for i in range(3):
        rgb[:, :, i] = normalize_to_uint8(rgb[:, :, i])
    return rgb

# 滑窗生成：依据 tile_size 和 overlap 生成切片范围
def tile_offsets(width, height, tile_size, overlap, max_tiles=None):
    step = max(1, tile_size - overlap)
    emitted = 0
    for y in range(0, height, step):
        h = min(tile_size, height - y)
        if h <= 0:
            continue
        for x in range(0, width, step):
            w = min(tile_size, width - x)
            if w <= 0:
                continue
            yield x, y, w, h
            emitted += 1
            if max_tiles is not None and emitted >= max_tiles:
                return

# 切片总数统计：用于进度显示和运行预估
def count_tiles(width, height, tile_size, overlap, max_tiles=None):
    step = max(1, tile_size - overlap)
    nx = (width + step - 1) // step
    ny = (height + step - 1) // step
    total = nx * ny
    if max_tiles is not None:
        return min(total, max_tiles)
    return total

# 坐标转换：将像素框转换为地理坐标框
def pixel_bbox_to_geo(transform, gx1, gy1, gx2, gy2):
    x_left, y_top = xy(transform, gy1, gx1, offset='ul')
    x_right, y_bottom = xy(transform, gy2, gx2, offset='ul')
    return box(min(x_left, x_right), min(y_top, y_bottom), max(x_left, x_right), max(y_top, y_bottom))

# 框重叠度计算：用于 NMS 判定重复检测
def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    if union <= 0:
        return 0.0
    return inter / union

# NMS 去重：按置信度和 IoU 移除重复框
def nms_bboxes(records, iou_thr):
    if not records:
        return []
    order = sorted(range(len(records)), key=lambda i: records[i]['conf'], reverse=True)
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        base = records[i]['bbox_px']
        remain = []
        for j in order:
            if bbox_iou(base, records[j]['bbox_px']) <= iou_thr:
                remain.append(j)
        order = remain
    return [records[i] for i in keep]

# 石块检测主流程：加载模型、切片预测、合并结果
def detect_stones(ortho_path, weights_path, tile_size, overlap, conf, iou_nms, max_tiles):
    log('Loading YOLO model')
    model = YOLO(weights_path)
    all_records = []
    with rasterio.open(ortho_path) as src:
        total_tiles = count_tiles(src.width, src.height, tile_size, overlap, max_tiles=max_tiles)
        log(f'Raster opened: width={src.width}, height={src.height}, tiles={total_tiles}')
        processed = 0
        for x, y, w, h in tile_offsets(src.width, src.height, tile_size, overlap, max_tiles=max_tiles):
            processed += 1
            tile = read_rgb_tile(src, x, y, w, h)
            results = model.predict(source=tile, conf=conf, verbose=False)
            if results and results[0].boxes is not None:
                for b in results[0].boxes:
                    x1, y1, x2, y2 = b.xyxy[0].tolist()
                    score = float(b.conf[0])
                    gx1 = x + float(x1)
                    gy1 = y + float(y1)
                    gx2 = x + float(x2)
                    gy2 = y + float(y2)
                    geom = pixel_bbox_to_geo(src.transform, gx1, gy1, gx2, gy2)
                    all_records.append({'conf': score, 'bbox_px': (gx1, gy1, gx2, gy2), 'geometry': geom})
            if processed == 1 or processed % 20 == 0 or processed == total_tiles:
                log(f'Detection progress: {processed}/{total_tiles} tiles, raw boxes={len(all_records)}')
        crs = src.crs
    log(f'Raw detections collected: {len(all_records)}')
    merged_records = nms_bboxes(all_records, iou_nms)
    log(f'Detections after cross-tile NMS: {len(merged_records)}')
    if not merged_records:
        return gpd.GeoDataFrame({'conf': []}, geometry=[], crs=crs)
    gdf = gpd.GeoDataFrame({'conf': [r['conf'] for r in merged_records]}, geometry=[r['geometry'] for r in merged_records], crs=crs)
    gdf = gdf[(~gdf.geometry.isna()) & (~gdf.geometry.is_empty)].copy()
    return gdf.reset_index(drop=True)

# 冠层擦除：用 canopy 图层裁掉石块重叠区域
def erase_canopy(stones, canopy_path):
    if len(stones) == 0:
        return stones
    log(f'Removing canopy overlap from {len(stones)} stones')
    canopy = gpd.read_file(canopy_path)
    if canopy.crs != stones.crs:
        canopy = canopy.to_crs(stones.crs)
    canopy_union = canopy.geometry.union_all() if hasattr(canopy.geometry, 'union_all') else canopy.geometry.unary_union
    out = stones.copy()
    out['geometry'] = out.geometry.apply(lambda g: g.difference(canopy_union))
    out = out[(~out.geometry.isna()) & (~out.geometry.is_empty)].copy()
    out = out.explode(index_parts=False).reset_index(drop=True)
    log(f'Stones after canopy erase: {len(out)}')
    return out

# 颜色到分值映射：根据主导色给类别和分数
def score_color(r, g, b):
    if g > r and g > b:
        return 'GREEN', 3
    if b > r and b > g:
        return 'BLUE', 2
    if r > g and r > b:
        return 'RED', 1
    return 'OTHER', 0

# 石块评分：采样植被栅格并计算单体与汇总分
def score_stones(stones, veg_path):
    if len(stones) == 0:
        return stones.copy(), {'count': 0, 'score_sum': 0.0, 'score_mean': 0.0}
    log(f'Scoring stones against vegetation raster: {len(stones)} features')
    with rasterio.open(veg_path) as veg:
        working = stones.to_crs(veg.crs) if stones.crs != veg.crs else stones.copy()
        coords = [(pt.x, pt.y) for pt in working.geometry.centroid]
        samples = list(veg.sample(coords))
        cmap = veg.colormap(1) if veg.count == 1 else None
        # 单波段且无 colormap 时，按采样值范围分档，避免全部打成 0 分
        if veg.count == 1 and (not cmap):
            single_vals = np.array([float(a[0]) for a in samples], dtype=np.float32)
            finite_vals = single_vals[np.isfinite(single_vals)]
            if finite_vals.size > 0:
                sv_min = float(np.min(finite_vals))
                sv_max = float(np.max(finite_vals))
            else:
                sv_min, sv_max = 0.0, 1.0
            if sv_max <= sv_min:
                sv_max = sv_min + 1.0
        cls_list = []
        score_list = []
        value_list = []
        for arr in samples:
            if veg.count >= 3:
                r, g, b = int(arr[0]), int(arr[1]), int(arr[2])
                label, score = score_color(r, g, b)
                value = None
            else:
                val = int(arr[0])
                value = val
                if cmap and val in cmap:
                    r, g, b, _ = cmap[val]
                    label, score = score_color(int(r), int(g), int(b))
                else:
                    # 无色表时按值域分三档：低/中/高 -> 1/2/3 分
                    norm = (float(val) - sv_min) / (sv_max - sv_min)
                    norm = max(0.0, min(1.0, norm))
                    if norm >= 0.66:
                        label, score = 'GREEN', 3
                    elif norm >= 0.33:
                        label, score = 'BLUE', 2
                    else:
                        label, score = 'RED', 1
            cls_list.append(label)
            score_list.append(score)
            value_list.append(value)
        scored = working.copy()
        scored['veg_cls'] = cls_list
        scored['score'] = score_list
        scored['veg_val'] = value_list
    if stones.crs != scored.crs:
        scored = scored.to_crs(stones.crs)
    summary = {'count': int(len(scored)), 'score_sum': float(np.sum(scored['score'])) if len(scored) else 0.0, 'score_mean': float(np.mean(scored['score'])) if len(scored) else 0.0}
    log(f"Scoring completed: count={summary['count']}, mean={summary['score_mean']:.3f}, sum={summary['score_sum']:.3f}")
    return scored, summary

# PTWL 风格配色：0-0.5 红到蓝，0.5-1 蓝到绿；0 为透明
def rgba_from_scores(scores):
    scores = np.asarray(scores, dtype=np.float32)
    rgba = np.zeros((scores.shape[0], 4), dtype=np.uint8)
    valid = scores > 0
    if not np.any(valid):
        return rgba
    values = np.clip(scores[valid], 0.0, 1.0)
    low = values <= 0.5
    t_low = np.zeros_like(values)
    t_low[low] = values[low] / 0.5
    rgba_valid = np.zeros((values.shape[0], 4), dtype=np.uint8)
    rgba_valid[low, 0] = np.rint(255.0 * (1.0 - t_low[low])).astype(np.uint8)
    rgba_valid[low, 2] = np.rint(255.0 * t_low[low]).astype(np.uint8)
    high = ~low
    t_high = np.zeros_like(values)
    t_high[high] = (values[high] - 0.5) / 0.5
    rgba_valid[high, 1] = np.rint(255.0 * t_high[high]).astype(np.uint8)
    rgba_valid[high, 2] = np.rint(255.0 * (1.0 - t_high[high])).astype(np.uint8)
    rgba_valid[:, 3] = 255
    rgba[valid] = rgba_valid
    return rgba

# Canopy 屏蔽：按网格与冠层重叠面积比例判定是否屏蔽
def calculate_canopy_mask(cells, canopy_path, overlap_threshold=0.2):
    blocked = np.zeros(len(cells), dtype=bool)
    canopy = gpd.read_file(canopy_path)
    canopy = canopy[canopy.geometry.notna() & (~canopy.geometry.is_empty)].copy()
    if len(canopy) == 0:
        return blocked
    if canopy.crs != cells.crs:
        canopy = canopy.to_crs(cells.crs)
    joined = gpd.sjoin(cells[['grid_id', 'geometry']], canopy[['geometry']], how='inner', predicate='intersects')
    if len(joined) == 0:
        return blocked
    intersections = joined.merge(canopy[['geometry']], left_on='index_right', right_index=True, how='left', suffixes=('', '_canopy'))
    overlap_areas = intersections.geometry.intersection(intersections['geometry_canopy']).area
    overlap_ratios = overlap_areas.groupby(intersections['grid_id']).sum() / intersections.groupby('grid_id').geometry.first().area
    blocked_ids = overlap_ratios.index[overlap_ratios.clip(upper=1.0) >= overlap_threshold].to_numpy()
    blocked[blocked_ids] = True
    return blocked

# 适宜性评分：不要 canopy + 高岩石密度 + 按植被色彩打分（仿照 PTWL）
def build_habitat_priority(density_grid, veg_path, canopy_path, canopy_overlap_threshold=0.2, vegetation_weight=0.7, rock_weight=0.3, rock_percentile=95.0):
    if len(density_grid) == 0:
        empty = density_grid.copy()
        empty['veg_sc'] = []
        empty['rock_sc'] = []
        empty['blocked'] = []
        empty['habitat_sc'] = []
        empty['priority'] = []
        return empty, {'cells': 0, 'blocked': 0, 'nonzero': 0, 'rock_cap': 1.0}
    working = density_grid.copy()
    blocked = calculate_canopy_mask(working, canopy_path, overlap_threshold=canopy_overlap_threshold)
    with rasterio.open(veg_path) as veg:
        sample_cells = working.to_crs(veg.crs) if working.crs != veg.crs else working.copy()
        coords = [(pt.x, pt.y) for pt in sample_cells.geometry.centroid]
        samples = list(veg.sample(coords))
        cmap = veg.colormap(1) if veg.count == 1 else None
        # 单波段且无 colormap 时，按采样值范围分档，避免 veg_sc 全 0
        if veg.count == 1 and (not cmap):
            single_vals = np.array([float(a[0]) for a in samples], dtype=np.float32)
            finite_vals = single_vals[np.isfinite(single_vals)]
            if finite_vals.size > 0:
                sv_min = float(np.min(finite_vals))
                sv_max = float(np.max(finite_vals))
            else:
                sv_min, sv_max = 0.0, 1.0
            if sv_max <= sv_min:
                sv_max = sv_min + 1.0
        veg_sc = []
        for arr in samples:
            if veg.count >= 3:
                _, score = score_color(int(arr[0]), int(arr[1]), int(arr[2]))
            else:
                val = int(arr[0])
                if cmap and val in cmap:
                    r, g, b, _ = cmap[val]
                    _, score = score_color(int(r), int(g), int(b))
                else:
                    norm = (float(val) - sv_min) / (sv_max - sv_min)
                    norm = max(0.0, min(1.0, norm))
                    score = 3 if norm >= 0.66 else (2 if norm >= 0.33 else 1)
            veg_sc.append(float(score) / 3.0)
    veg_sc = np.array(veg_sc, dtype=np.float32)
    dens = np.asarray(working['dens_ha'], dtype=np.float32)
    positive = dens[dens > 0]
    rock_cap = float(np.percentile(positive, rock_percentile)) if positive.size > 0 else 1.0
    if rock_cap <= 0:
        rock_cap = 1.0
    rock_sc = np.clip(dens / rock_cap, 0.0, 1.0)
    eligible = (~blocked) & (dens > 0)
    habitat = ((veg_sc * vegetation_weight) + (rock_sc * rock_weight)) / (vegetation_weight + rock_weight)
    habitat = np.where(eligible, habitat, 0.0).astype(np.float32)
    # 多档位分级：把适宜性分数映射到更细的等级标签
    labels = np.where(habitat >= 0.85, 'S1_VERY_HIGH', np.where(habitat >= 0.70, 'S2_HIGH', np.where(habitat >= 0.55, 'S3_MEDIUM_HIGH', np.where(habitat >= 0.40, 'S4_MEDIUM', np.where(habitat >= 0.25, 'S5_LOW', np.where(habitat > 0, 'S6_VERY_LOW', 'NODATA'))))))
    out = working.copy()
    out['veg_sc'] = veg_sc
    out['rock_sc'] = rock_sc.astype(np.float32)
    out['blocked'] = blocked.astype(np.int16)
    out['habitat_sc'] = habitat
    out['priority'] = labels
    summary = {'cells': int(len(out)), 'blocked': int(np.sum(blocked)), 'nonzero': int(np.sum(habitat > 0)), 'rock_cap': float(rock_cap)}
    log(f"Habitat priority completed: cells={summary['cells']}, blocked={summary['blocked']}, nonzero={summary['nonzero']}, rock_cap={summary['rock_cap']:.3f}")
    return out, summary

# 专题图输出：保存适宜性评分网格图（PNG）
def save_habitat_png(habitat_grid, canopy_path, out_dir):
    fig_path = os.path.join(out_dir, 'habitat_priority_map.png')
    fig, ax = plt.subplots(figsize=(10, 10), dpi=180)
    if len(habitat_grid) == 0:
        ax.set_title('Habitat Priority Map (empty)')
        ax.axis('off')
        fig.savefig(fig_path, bbox_inches='tight')
        plt.close(fig)
        return fig_path
    # 仅绘制有效适宜性区域：不适合/无石头区域作为 NoData 不显示
    valid_grid = habitat_grid[habitat_grid['habitat_sc'] > 0].copy()
    if len(valid_grid) > 0:
        rgba = rgba_from_scores(valid_grid['habitat_sc'].to_numpy(dtype=np.float32))
        colors = [tuple(c / 255.0) for c in rgba]
        valid_grid.plot(ax=ax, color=colors, edgecolor='none')
    canopy = gpd.read_file(canopy_path)
    canopy = canopy[(~canopy.geometry.isna()) & (~canopy.geometry.is_empty)].copy()
    if len(canopy) > 0:
        if canopy.crs != habitat_grid.crs:
            canopy = canopy.to_crs(habitat_grid.crs)
        canopy.boundary.plot(ax=ax, color='black', linewidth=0.5, alpha=0.8)
    ax.set_title('Habitat Priority (No Canopy + Rock Density + Vegetation Score)')
    ax.set_axis_off()
    fig.savefig(fig_path, bbox_inches='tight')
    plt.close(fig)
    return fig_path

# 密度分析：生成网格并统计每格石块密度
def build_density_grid(stones, grid_size_m):
    log('Building density grid')
    if len(stones) == 0:
        return gpd.GeoDataFrame({'Join_Count': [], 'area_ha': [], 'dens_ha': []}, geometry=[], crs=stones.crs)
    src_crs = stones.crs
    working = stones
    if getattr(working.crs, 'is_geographic', False):
        working = working.to_crs(working.estimate_utm_crs())
    minx, miny, maxx, maxy = working.total_bounds
    xs = np.arange(minx, maxx + grid_size_m, grid_size_m)
    ys = np.arange(miny, maxy + grid_size_m, grid_size_m)
    cells = [box(xs[i], ys[j], xs[i + 1], ys[j + 1]) for i in range(len(xs) - 1) for j in range(len(ys) - 1)]
    grid = gpd.GeoDataFrame({'grid_id': np.arange(len(cells))}, geometry=cells, crs=working.crs)
    joined = gpd.sjoin(grid, working[['geometry']], how='left', predicate='intersects')
    valid = joined[joined['index_right'].notna()]
    counts = valid.groupby('grid_id').size()
    grid['Join_Count'] = grid['grid_id'].map(counts).fillna(0).astype(int)
    grid['area_ha'] = grid.geometry.area / 10000.0
    grid['dens_ha'] = np.where(grid['area_ha'] > 0, grid['Join_Count'] / grid['area_ha'], 0.0)
    if src_crs != grid.crs:
        grid = grid.to_crs(src_crs)
    log(f'Density grid completed: cells={len(grid)}')
    return grid

# 清理旧输出：删除同名 shapefile 相关文件
def remove_shapefile(path):
    stem, _ = os.path.splitext(path)
    for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg', '.qix', '.fix']:
        p = stem + ext
        if os.path.exists(p):
            os.remove(p)

# 保存结果：按子文件夹分类输出（vectors/tables）
def save_outputs(out_dir, stones_raw, stones_nocanopy, stones_scored, density_grid, habitat_grid, score_summary, habitat_summary):
    vector_dir = os.path.join(out_dir, 'vectors')
    table_dir = os.path.join(out_dir, 'tables')
    os.makedirs(vector_dir, exist_ok=True)
    os.makedirs(table_dir, exist_ok=True)
    raw_path = os.path.join(vector_dir, 'stones_raw.shp')
    nocanopy_path = os.path.join(vector_dir, 'stones_nocanopy.shp')
    scored_path = os.path.join(vector_dir, 'stones_scored.shp')
    density_path = os.path.join(vector_dir, 'stone_density_grid.shp')
    habitat_path = os.path.join(vector_dir, 'habitat_priority_grid.shp')
    score_csv = os.path.join(table_dir, 'stone_score_summary.csv')
    habitat_csv = os.path.join(table_dir, 'habitat_priority_summary.csv')
    for shp in [raw_path, nocanopy_path, scored_path, density_path, habitat_path]:
        remove_shapefile(shp)
    stones_raw.to_file(raw_path, driver='ESRI Shapefile')
    stones_nocanopy.to_file(nocanopy_path, driver='ESRI Shapefile')
    stones_scored.to_file(scored_path, driver='ESRI Shapefile')
    density_grid.to_file(density_path, driver='ESRI Shapefile')
    habitat_grid.to_file(habitat_path, driver='ESRI Shapefile')
    pd.DataFrame([score_summary]).to_csv(score_csv, index=False)
    pd.DataFrame([habitat_summary]).to_csv(habitat_csv, index=False)
    log('Outputs saved')
    log(f' - {raw_path}')
    log(f' - {nocanopy_path}')
    log(f' - {scored_path}')
    log(f' - {density_path}')
    log(f' - {habitat_path}')
    log(f' - {score_csv}')
    log(f' - {habitat_csv}')
    return raw_path, nocanopy_path, scored_path, density_path, habitat_path, score_csv, habitat_csv

# 总控函数：串联执行检测、擦除、评分、密度、适宜性建图与保存
def run_pipeline(ortho, canopy, veg, weights, out_dir, tile_size, overlap, conf, iou_nms, grid_size_m, canopy_overlap_threshold=0.2, vegetation_weight=0.7, rock_weight=0.3, rock_percentile=95.0, max_tiles=None):
    ensure_workspace(out_dir)
    stones_raw = detect_stones(ortho, weights, tile_size, overlap, conf, iou_nms, max_tiles)
    log(f'Raw stone features: {len(stones_raw)}')
    stones_nocanopy = erase_canopy(stones_raw, canopy)
    stones_scored, score_summary = score_stones(stones_nocanopy, veg)
    density_grid = build_density_grid(stones_nocanopy, grid_size_m)
    habitat_grid, habitat_summary = build_habitat_priority(
        density_grid=density_grid,
        veg_path=veg,
        canopy_path=canopy,
        canopy_overlap_threshold=canopy_overlap_threshold,
        vegetation_weight=vegetation_weight,
        rock_weight=rock_weight,
        rock_percentile=rock_percentile,
    )
    return save_outputs(out_dir, stones_raw, stones_nocanopy, stones_scored, density_grid, habitat_grid, score_summary, habitat_summary)


In [ ]:
# 执行入口：自动识别文件夹中的 shp/tif/图像/权重文件并运行管线
def resolve_input_path(path_or_dir, allowed_exts, label):
    p = os.path.expanduser(path_or_dir)
    if os.path.isfile(p):
        ext = os.path.splitext(p)[1].lower()
        if ext not in allowed_exts:
            raise ValueError(f'{label} file type not supported: {p}')
        return p
    if not os.path.isdir(p):
        raise FileNotFoundError(f'{label} path not found: {p}')
    rank = {ext: i for i, ext in enumerate(allowed_exts)}
    candidates = []
    for root, _, files in os.walk(p):
        for name in files:
            ext = os.path.splitext(name)[1].lower()
            if ext in rank:
                full = os.path.join(root, name)
                candidates.append((rank[ext], full))
    if not candidates:
        raise FileNotFoundError(f'No supported {label} file found under: {p}')
    candidates.sort(key=lambda x: (x[0], x[1]))
    chosen = candidates[0][1]
    print(f'Auto-selected {label}: {chosen}')
    return chosen

# 1) 自动解析输入（可填文件或文件夹）
ortho_path = resolve_input_path(ORTHO, ['.tif', '.tiff', '.img', '.jp2', '.vrt', '.png', '.jpg', '.jpeg', '.webp'], 'ORTHO')
veg_path = resolve_input_path(VEG, ['.tif', '.tiff', '.img', '.jp2', '.vrt', '.png', '.jpg', '.jpeg', '.webp'], 'VEG')
weights_path = resolve_input_path(WEIGHTS, ['.pt', '.pth', '.onnx'], 'WEIGHTS')
canopy_path = resolve_input_path(CANOPY, ['.shp'], 'CANOPY')

# 2) CANOPY shapefile 组件完整性检查
canopy_base, _ = os.path.splitext(canopy_path)
required_canopy_parts = ['.shp', '.shx', '.dbf']
optional_canopy_parts = ['.prj', '.cpg']
missing_required = [canopy_base + ext for ext in required_canopy_parts if not os.path.exists(canopy_base + ext)]
missing_optional = [canopy_base + ext for ext in optional_canopy_parts if not os.path.exists(canopy_base + ext)]
if missing_required:
    msg = (
        'CANOPY shapefile is incomplete. Missing required sidecar files:\n'
        + '\n'.join(missing_required)
        + '\nPlease upload the full shapefile set with the same basename.'
    )
    raise FileNotFoundError(msg)
if missing_optional:
    print('Warning: optional canopy sidecar files missing:')
    for p in missing_optional:
        print(' -', p)

# 3) 通过检查后开始执行

effective_max_tiles = 1 if SMOKE_TEST and MAX_TILES is None else MAX_TILES
log('Pipeline started')
outputs = run_pipeline(
    ortho=ortho_path,
    canopy=canopy_path,
    veg=veg_path,
    weights=weights_path,
    out_dir=OUT_DIR,
    tile_size=TILE_SIZE,
    overlap=OVERLAP,
    conf=CONF,
    iou_nms=IOU_NMS,
    grid_size_m=GRID_SIZE_M,
    canopy_overlap_threshold=CANOPY_OVERLAP_THRESHOLD,
    vegetation_weight=VEGETATION_WEIGHT,
    rock_weight=ROCK_WEIGHT,
    rock_percentile=ROCK_PERCENTILE,
    max_tiles=effective_max_tiles,
)
log('Pipeline finished')
print('Completed')
for p in outputs:
    print(p)